___
<a href='https://honglab.ai'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 엔그램으로 단어 만들기

**해당 강의:** 파트1 챕터2 「04-03. 바이그램」 · 「04-04. 트라이그램」 · 「04-05. N-그램」

앞 글자를 보고 다음 글자를 고르는 n-그램 모델로 단어를 만듭니다. 만든 단어 중 실제 사전에 있는 단어가 얼마나 되는지 세어서, n에 따라 무엇이 달라지는지 비교합니다.

- 실행 시간: 20초 안팎 (CPU)
- 메모리: 1GB 이하
- 설치할 것: 없음 (파이썬 표준 라이브러리만 씁니다)

In [1]:
import random
from collections import Counter, defaultdict

words = [line.strip() for line in open("data/words.txt", encoding="utf-8") if line.strip()]
wordset = set(words)          # 만든 단어가 사전에 있는지 확인할 때 씁니다
print(f"단어 수: {len(words):,}")

단어 수: 420,309


## 슬라이딩 윈도우로 세기

두 글자짜리 창을 한 칸씩 밀면서 글자 쌍을 셉니다. 강의에서 본 세 단어로 먼저 해 봅니다.

In [2]:
toy = ["가가", "가가대소", "가가대소하다"]
toy_bigrams = Counter()
for w in toy:
    for i in range(len(w) - 1):
        toy_bigrams[w[i:i + 2]] += 1

for pair, cnt in toy_bigrams.items():
    print(pair, cnt)

가가 3
가대 2
대소 2
소하 1
하다 1


「가」 다음에는 「가」가 3번, 「대」가 2번 왔습니다. 그래서 「가」 다음 글자로는 「가」의 확률이 더 높습니다.
이처럼 앞 글자가 정해졌을 때의 다음 글자 확률을 조건부 확률 P(다음 글자 | 앞 글자)라고 합니다.

## 시작과 끝 표시: BOS와 EOS

단어를 만들려면 어떤 글자로 시작하고 어디서 끝나는지도 알아야 합니다.
그래서 단어 앞에는 시작 표시 `<BOS>`를, 뒤에는 끝 표시 `<EOS>`를 붙여서 함께 셉니다.
n-그램은 앞의 n−1글자를 보므로 `<BOS>`를 n−1개 붙입니다.

In [3]:
BOS, EOS = "<BOS>", "<EOS>"

def pad(word, n):
    return [BOS] * (n - 1) + list(word) + [EOS]

print(pad("나비", 2))
print(pad("나비", 3))

['<BOS>', '나', '비', '<EOS>']
['<BOS>', '<BOS>', '나', '비', '<EOS>']


## n-그램 표 만들기

문맥(앞의 n−1글자)마다 다음 글자가 몇 번 왔는지 저장합니다. 파이썬 딕셔너리가 강의에서 말한 해시 테이블입니다.

In [4]:
def count_ngrams(words, n):
    table = defaultdict(Counter)        # 문맥(앞 n-1글자) → Counter(다음 글자: 횟수)
    for w in words:
        toks = pad(w, n)
        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            table[context][toks[i + n - 1]] += 1
    return table

bigram = count_ngrams(words, 2)
print(f"문맥 수: {len(bigram):,}")
print(f"고유 바이그램 수: {sum(len(c) for c in bigram.values()):,}\n")

pairs = Counter({(ctx[0], nxt): cnt for ctx, c in bigram.items() for nxt, cnt in c.items()})
for (a, b), cnt in pairs.most_common(5):
    print(f"{a} → {b}: {cnt:,}")

문맥 수: 2,451
고유 바이그램 수: 171,425

다 → <EOS>: 71,419
하 → 다: 44,128
기 → <EOS>: 11,896
이 → <EOS>: 9,442
리 → <EOS>: 7,758


가장 많은 쌍은 「다 → <EOS>」와 「하 → 다」입니다. 「다」로 끝나는 동사·형용사, 그중에서도 「-하다」로 끝나는 표제어가 많기 때문입니다.

「가」 다음에 올 글자의 확률도 봅니다.

In [5]:
after = bigram[("가",)]
n_after = sum(after.values())
for nxt, cnt in after.most_common(5):
    print(f"P({nxt} | 가) = {cnt:,} / {n_after:,} = {cnt / n_after:.3f}")

P(<EOS> | 가) = 1,848 / 12,270 = 0.151
P(지 | 가) = 478 / 12,270 = 0.039
P(리 | 가) = 402 / 12,270 = 0.033
P(스 | 가) = 368 / 12,270 = 0.030
P(락 | 가) = 329 / 12,270 = 0.027


## 단어 만들기

`<BOS>`에서 시작해 표에서 다음 글자를 확률대로 뽑고, `<EOS>`가 나오면 멈춥니다.
사전에 있는 단어면 `[O]`, 없으면 `[X]`를 붙입니다.

In [6]:
def generate(table, n, rng, max_len=30):
    context = [BOS] * (n - 1)
    out = []
    while len(out) < max_len:
        counter = table[tuple(context[-(n - 1):])]
        nxt = rng.choices(list(counter), weights=list(counter.values()))[0]
        if nxt == EOS:
            break
        out.append(nxt)
        context.append(nxt)
    return "".join(out)

rng = random.Random(0)
for _ in range(15):
    w = generate(bigram, 2, rng)
    print(w, "[O]" if w in wordset else "[X]")

초해안전하다 [X]
멸치르흐래 [X]
정장 [O]
퍼코카풀 [X]
재검도계 [X]
사서론치기남동식하다 [X]
부주군 [X]
소륵 [X]
문잔 [X]
암 [O]
호재론 [X]
무량초 [X]
아민복가마정신청운야정맥 [X]
다 [O]
영광체 [X]


## 탐욕적 디코딩

매번 가장 많이 나온 글자만 고르면 어떻게 될까요?

In [7]:
def greedy(table, n, max_len=30):
    context = [BOS] * (n - 1)
    out = []
    while len(out) < max_len:
        nxt = table[tuple(context[-(n - 1):])].most_common(1)[0][0]
        if nxt == EOS:
            break
        out.append(nxt)
        context.append(nxt)
    return "".join(out)

print([greedy(bigram, 2) for _ in range(5)])
print("<BOS> 다음:", bigram[(BOS,)].most_common(3))
print("「사」 다음:", bigram[("사",)].most_common(3))

['사', '사', '사', '사', '사']
<BOS> 다음: [('사', 4614), ('가', 4448), ('자', 4105)]
「사」 다음: [('<EOS>', 4539), ('회', 583), ('이', 438)]


몇 번을 만들어도 「사」 한 글자짜리 단어만 나옵니다. 「사」로 시작하는 표제어가 가장 많고, 「사」 다음에는 단어가 끝나는 경우가 가장 많기 때문입니다.

## 트라이그램

앞의 두 글자를 봅니다. 함수는 그대로 두고 n만 3으로 바꿉니다.

In [8]:
trigram = count_ngrams(words, 3)
print(f"고유 트라이그램 수: {sum(len(c) for c in trigram.values()):,}\n")

rng = random.Random(0)
for _ in range(15):
    w = generate(trigram, 3, rng)
    print(w, "[O]" if w in wordset else "[X]")

고유 트라이그램 수: 654,039

초자식 [X]
수상지 [X]
선유하다 [O]
정온식물피복조개자반증 [X]
군사시창하다 [X]
란타넘 [O]
잘라먹다 [O]
소청 [O]
문한 [O]
암맹하다 [O]
관리사납다 [X]
자생적방어구이 [X]
엘리아 [X]
다듬이질하다 [O]
클린업하다 [O]


## 1,000개로 비교하기: 사전 일치율

몇 개를 눈으로 봐서는 어느 쪽이 나은지 판단하기 어렵습니다. 1,000개를 만들어서 사전에 있는 단어의 비율을 셉니다.

In [9]:
def match_rate(table, n, k=1000, seed=0):
    rng = random.Random(seed)
    made = [generate(table, n, rng) for _ in range(k)]
    return sum(w in wordset for w in made) / k, made

for name, table, n in [("바이그램", bigram, 2), ("트라이그램", trigram, 3)]:
    rate, _ = match_rate(table, n)
    print(f"{name}: {rate:.1%}")

del bigram, trigram

바이그램: 41.1%
트라이그램: 56.7%


## n을 2에서 6까지

같은 함수로 n만 바꿔 가며 비교합니다. 사전에 없는 단어, 즉 새로 만든 단어도 몇 개씩 봅니다.

In [10]:
rows = []
for n in range(2, 7):
    table = count_ngrams(words, n)
    rate, made = match_rate(table, n)
    new_words = [w for w in made if w not in wordset]
    rows.append((n, sum(len(c) for c in table.values()), len(table), rate, new_words))
    if n == 6:
        six = table                     # 아래에서 한 번 더 씁니다

print("n | 고유 n-그램 | 문맥 수 | 사전 일치율")
for n, n_grams, n_contexts, rate, _ in rows:
    print(f"{n} | {n_grams:>11,} | {n_contexts:>7,} | {rate:.1%}")

print()
for n, *_, new_words in rows:
    print(f"{n}-그램이 새로 만든 단어 ({len(new_words)}개):", ", ".join(new_words[:6]) or "없음")

n | 고유 n-그램 | 문맥 수 | 사전 일치율
2 |     171,425 |   2,451 | 41.1%
3 |     654,039 | 169,589 | 56.7%
4 |     976,877 | 547,576 | 94.6%
5 |   1,105,860 | 684,407 | 99.2%
6 |   1,141,426 | 720,877 | 99.8%

2-그램이 새로 만든 단어 (589개): 초해안전하다, 멸치르흐래, 퍼코카풀, 재검도계, 사서론치기남동식하다, 부주군
3-그램이 새로 만든 단어 (433개): 초자식, 수상지, 정온식물피복조개자반증, 군사시창하다, 관리사납다, 자생적방어구이
4-그램이 새로 만든 단어 (54개): 집합명사채, 염화티탄, 아이에프오에이치클럽, 니켈축전지기관차끌힘, 지리터리, 달라디에르
5-그램이 새로 만든 단어 (8개): 클라이나이트로펜톤, 에이프런스테이지댄스, 스페이스오프, 메노알레그로주스토, 네델란드부르죠아제도, 집단히스테리시스
6-그램이 새로 만든 단어 (2개): 클라이나이트로벤졸, 군위아미타여래좌상


- n=2–3: 사전에 없는 어색한 단어가 많습니다.
- n=4–5: 대부분 사전에 있는 단어이고, 새로 만든 단어도 있을 법한 모양입니다.
- n=6: 거의 모든 단어가 사전에 있습니다. 새 단어를 만들지 못하고 사전을 외워서 읊습니다.

n이 작으면 짧은 패턴밖에 담지 못하고, n이 크면 훈련 데이터를 그대로 외웁니다. 일반화(generalization)와 암기(memorization) 사이의 트레이드오프입니다.

## 선택지가 사라지는 과정

6-그램은 앞의 다섯 글자를 봅니다. 「자동판매기」를 한 글자씩 만드는 동안, 매번 다음 글자의 후보가 몇 개인지 셉니다.

In [11]:
print(f"표제어 평균 길이: {sum(map(len, words)) / len(words):.1f}글자\n")

word = "자동판매기"
toks = pad(word, 6)
for k in range(len(word) + 1):
    counter = six[tuple(toks[k:k + 5])]
    so_far = word[:k] or "(시작)"
    examples = [x for x, _ in counter.most_common(4)]
    print(f"「{so_far}」 다음 후보: {len(counter):,}개   예: {examples}")

표제어 평균 길이: 3.7글자

「(시작)」 다음 후보: 2,189개   예: ['사', '가', '자', '고']
「자」 다음 후보: 431개   예: ['기', '연', '동', '유']
「자동」 다음 후보: 118개   예: ['차', '조', '제', '추']
「자동판」 다음 후보: 1개   예: ['매']
「자동판매」 다음 후보: 1개   예: ['기']
「자동판매기」 다음 후보: 1개   예: ['<EOS>']


「자동판」까지 정해지면 다음 후보가 하나뿐입니다. 표제어의 평균 길이가 4글자가 안 되므로, 서너 글자만 정해져도 그 글자로 시작하는 표제어가 한두 개로 줄고 나머지 글자가 거의 정해집니다.
n이 단어 길이만큼 커지면 사전에 있는 단어를 그대로 따라갈 수밖에 없습니다.

## 메모리: 가능한 조합과 실제로 나온 조합

글자가 2,450종이면 가능한 n-그램은 2,450의 n제곱개입니다. 하지만 표에는 데이터에 실제로 나온 조합만 저장하므로, n을 키워도 표의 항목 수는 데이터의 길이를 넘지 않습니다.

In [12]:
n_chars = len({ch for w in words for ch in w})
for n, n_grams, *_ in rows:
    print(f"{n}-그램   가능한 조합 {n_chars ** n:.1e}   실제로 나온 조합 {n_grams:,}")

2-그램   가능한 조합 6.0e+06   실제로 나온 조합 171,425
3-그램   가능한 조합 1.5e+10   실제로 나온 조합 654,039
4-그램   가능한 조합 3.6e+13   실제로 나온 조합 976,877
5-그램   가능한 조합 8.8e+16   실제로 나온 조합 1,105,860
6-그램   가능한 조합 2.2e+20   실제로 나온 조합 1,141,426


## 정리

- n-그램은 앞의 n−1글자를 문맥으로 삼아 다음 글자의 횟수를 세고, 그 비율로 다음 글자를 뽑습니다.
- n이 커질수록 사전에 있는 단어를 더 많이 만들지만, 너무 크면 사전을 외워서 새 단어를 만들지 못합니다.
- 표에는 실제로 나온 조합만 저장합니다.

`03_엔그램으로문장만들기`에서는 소설로 문장을 만듭니다.